# 02. Leakage-safe LightGBM Baseline

EDA에서 얻은 인사이트를 반영한 LightGBM 모델입니다. 모든 데이터 의존 전처리는 각 Fold의 학습 구간에만 fit하며, 단일 무작위 검증과 시간 기반 다중 검증을 함께 사용합니다.

- 결측치 중앙값: 각 Fold 학습 데이터에서만 계산
- 범주 인코더: 각 Fold 학습 데이터에서만 fit
- 미지 범주: `OneHotEncoder(handle_unknown='ignore')`로 처리
- 시간 검증: 2025년 2·3·4분기를 순서대로 검증하는 expanding window
- Target으로 만든 변수는 사용하지 않음
- RMSLE와 원가격 RMSE를 모두 출력

In [1]:
from pathlib import Path
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')
RANDOM_STATE = 42

In [2]:
candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p for p in candidates if (p / 'data').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('프로젝트 루트를 찾을 수 없습니다.')

DATA_DIR = PROJECT_ROOT / 'data'
SUBMISSION_DIR = PROJECT_ROOT / 'submissions'
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(DATA_DIR / 'seoul_real_estate_train.csv')
test = pd.read_csv(DATA_DIR / 'seoul_real_estate_test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print('Train:', train.shape)
print('Test :', test.shape)
print('Train period:', train['Transaction_YearMonth'].min(), '~', train['Transaction_YearMonth'].max())
print('Test period :', test['Transaction_YearMonth'].min(), '~', test['Transaction_YearMonth'].max())

Train: (1969, 11)
Test : (531, 10)
Train period: 202401 ~ 202512
Test period : 202601 ~ 202603


## 1. 평가 함수

In [3]:
def rmse(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def rmsle(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.clip(np.asarray(y_pred), 0, None)
    return rmse(np.log1p(y_true), np.log1p(y_pred))

## 2. 누수 없는 전처리

`fit_feature_transformer()`는 중앙값과 범주 목록을 오직 `fit_df`에서만 학습합니다. 검증 또는 Test에 처음 등장하는 범주는 모두 0인 one-hot 벡터로 안전하게 처리합니다.

In [4]:
CATEGORICAL_COLS = ['Gu', 'Dong']

def engineer_features(df, subway_median):
    x = df.copy()
    x['Transaction_Year'] = x['Transaction_YearMonth'] // 100
    x['Transaction_Month'] = x['Transaction_YearMonth'] % 100
    x['Time_Index'] = (x['Transaction_Year'] - 2024) * 12 + x['Transaction_Month'] - 1
    x['Age_at_Transaction'] = x['Transaction_Year'] - x['Year_Built']

    x['Subway_Distance_Missing'] = x['Distance_to_Subway'].isna().astype('int8')
    x['Distance_to_Subway'] = x['Distance_to_Subway'].fillna(subway_median)

    x['Log_Area'] = np.log(x['Exclusive_Area'])
    x['Area_Squared'] = x['Exclusive_Area'] ** 2
    x['Area_Cubed_Scaled'] = x['Exclusive_Area'] ** 3 / 10_000
    x['Floor_Squared'] = x['Floor'] ** 2
    x['Distance_Squared'] = x['Distance_to_Subway'] ** 2
    x['Age_Squared'] = x['Age_at_Transaction'] ** 2
    x['Time_Squared'] = x['Time_Index'] ** 2

    return x.drop(columns=['ID', 'Target', 'Transaction_YearMonth', 'Year_Built'], errors='ignore')

def fit_feature_transformer(fit_df):
    subway_median = fit_df['Distance_to_Subway'].median()
    fit_raw = engineer_features(fit_df, subway_median)
    numeric_cols = [c for c in fit_raw.columns if c not in CATEGORICAL_COLS]
    encoder = OneHotEncoder(
        handle_unknown='ignore', sparse_output=False, dtype=np.int8
    )
    encoder.fit(fit_raw[CATEGORICAL_COLS])

    def transform(df):
        raw = engineer_features(df, subway_median)
        numeric = raw[numeric_cols].reset_index(drop=True)
        encoded = pd.DataFrame(
            encoder.transform(raw[CATEGORICAL_COLS]),
            columns=encoder.get_feature_names_out(CATEGORICAL_COLS),
        )
        return pd.concat([numeric, encoded], axis=1)

    metadata = {
        'subway_median': subway_median,
        'categories': {
            col: list(categories)
            for col, categories in zip(CATEGORICAL_COLS, encoder.categories_)
        },
    }
    return transform, metadata

## 3. 모델 설정

In [5]:
MODEL_PARAMS = {
    'objective': 'regression',
    'metric': 'rmse',
    'n_estimators': 3000,
    'learning_rate': 0.015,
    'num_leaves': 4,
    'min_child_samples': 40,
    'feature_fraction': 1.0,
    'bagging_fraction': 1.0,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'linear_tree': True,
    'linear_lambda': 10.0,
    'random_state': RANDOM_STATE,
    'verbosity': -1,
    'n_jobs': 1,
}

def fit_and_evaluate(fit_df, valid_df, fold_name):
    transform, metadata = fit_feature_transformer(fit_df)
    X_fit = transform(fit_df)
    X_valid = transform(valid_df)
    y_fit_log = np.log1p(fit_df['Target'])
    y_valid_log = np.log1p(valid_df['Target'])

    model = lgb.LGBMRegressor(**MODEL_PARAMS)
    model.fit(
        X_fit, y_fit_log,
        eval_set=[(X_valid, y_valid_log)],
        callbacks=[lgb.early_stopping(150, verbose=False)],
    )
    pred = np.clip(np.expm1(model.predict(X_valid)), 0, None)
    result = {
        'Fold': fold_name,
        'Train_rows': len(fit_df),
        'Valid_rows': len(valid_df),
        'RMSE': rmse(valid_df['Target'], pred),
        'RMSLE': rmsle(valid_df['Target'], pred),
        'Best_iteration': model.best_iteration_,
        'Subway_median': metadata['subway_median'],
    }
    return model, result

## 4. 무작위 Holdout 검증

기존 모델과의 참고 비교를 위한 검증입니다. 전처리기는 80% 학습 구간에만 fit됩니다. 실제 Test 구조에는 다음 절의 시간 검증이 더 가깝습니다.

In [6]:
random_train_idx, random_valid_idx = train_test_split(
    np.arange(len(train)), test_size=0.2, random_state=RANDOM_STATE
)
random_fit = train.iloc[random_train_idx].copy()
random_valid = train.iloc[random_valid_idx].copy()
random_model, random_result = fit_and_evaluate(
    random_fit, random_valid, 'Random 80:20'
)
display(pd.DataFrame([random_result]))

,Fold,Train_rows,Valid_rows,RMSE,RMSLE,Best_iteration,Subway_median
0,Random 80:20,1575,394,"2,596.380417",0.079210,1699,0.275000


## 5. 시간 기반 다중 검증

각 Fold는 검증 분기보다 과거인 데이터만 학습합니다. Fold가 진행될수록 학습 구간이 확장되며, 동일 거래월이 학습과 검증에 동시에 들어가지 않습니다.

| Fold | 학습 기간 | 검증 기간 |
|---|---|---|
| 1 | 2024-01 ~ 2025-03 | 2025-04 ~ 2025-06 |
| 2 | 2024-01 ~ 2025-06 | 2025-07 ~ 2025-09 |
| 3 | 2024-01 ~ 2025-09 | 2025-10 ~ 2025-12 |

In [7]:
TIME_FOLDS = [
    ('2025 Q2', 202504, 202507),
    ('2025 Q3', 202507, 202510),
    ('2025 Q4', 202510, 202601),
]

time_results = []
time_models = []
for fold_name, valid_start, valid_end in TIME_FOLDS:
    fit_mask = train['Transaction_YearMonth'] < valid_start
    valid_mask = train['Transaction_YearMonth'].between(
        valid_start, valid_end - 1
    )
    fold_fit = train.loc[fit_mask].copy()
    fold_valid = train.loc[valid_mask].copy()

    assert fold_fit['Transaction_YearMonth'].max() < fold_valid['Transaction_YearMonth'].min()
    model, result = fit_and_evaluate(fold_fit, fold_valid, fold_name)
    time_models.append(model)
    time_results.append(result)

time_scores = pd.DataFrame(time_results)
display(time_scores)
print(f"Mean time RMSE : {time_scores['RMSE'].mean():,.2f}")
print(f"Mean time RMSLE: {time_scores['RMSLE'].mean():.6f}")
print(f"RMSE std       : {time_scores['RMSE'].std(ddof=0):,.2f}")

,Fold,Train_rows,Valid_rows,RMSE,RMSLE,Best_iteration,Subway_median
0,2025 Q2,1201,278,"2,121.340569",0.056061,1475,0.259500
1,2025 Q3,1479,244,"2,647.815144",0.081294,1363,0.264000
2,2025 Q4,1723,246,"2,718.178533",0.076348,1614,0.266000


Mean time RMSE : 2,495.78
Mean time RMSLE: 0.071234
RMSE std       : 266.32


## 6. 전체 데이터 재학습 및 제출

최종 전처리기는 전체 Train에만 fit하고 Test에는 transform만 수행합니다. 부스팅 반복 횟수는 단일 Holdout에 종속되지 않도록 세 개 시간 Fold의 최적 반복 횟수 중앙값을 사용합니다.

In [8]:
final_transform, final_metadata = fit_feature_transformer(train)
X_full = final_transform(train)
X_test = final_transform(test)
y_full_log = np.log1p(train['Target'])

final_rounds = int(np.median(time_scores['Best_iteration']))
final_params = MODEL_PARAMS.copy()
final_params['n_estimators'] = final_rounds

final_model = lgb.LGBMRegressor(**final_params)
final_model.fit(X_full, y_full_log)
test_pred = np.clip(np.expm1(final_model.predict(X_test)), 0, None)

prediction_frame = pd.DataFrame({'ID': test['ID'], 'Target': test_pred})
submission = sample_submission[['ID']].merge(
    prediction_frame, on='ID', how='left', validate='one_to_one'
)
assert submission['Target'].notna().all()
assert len(submission) == len(sample_submission)
assert list(X_full.columns) == list(X_test.columns)

output_path = SUBMISSION_DIR / '02_baseline_lgbm_submission.csv'
submission.to_csv(output_path, index=False)
print(f'Final boosting rounds: {final_rounds}')
print(f'Full-train subway median: {final_metadata["subway_median"]:.3f}')
print(f'Saved: {output_path}')
display(submission.head())
display(submission['Target'].describe())

Final boosting rounds: 1475
Full-train subway median: 0.273
Saved: /Users/joyeojin/10_Dev/11_Projects/korea-apartment-price-prediction/submissions/02_baseline_lgbm_submission.csv


,ID,Target
0,TR_2427,"36,525.734357"
1,TR_0028,"47,675.835331"
2,TR_1461,"28,566.220288"
3,TR_1977,"46,907.389664"
4,TR_2351,"46,070.204919"


count      531.000000
mean    39,802.794087
std      9,835.140746
min     14,468.716678
25%     32,856.984928
50%     39,204.799049
75%     46,706.302028
max     67,736.889441
Name: Target, dtype: float64

## 누수 방지 체크리스트

- [x] Target 기반 파생변수 미사용
- [x] 결측치 대체값을 각 Fold 학습 구간에서만 계산
- [x] 범주 인코더를 각 Fold 학습 구간에서만 fit
- [x] 검증 시점보다 미래인 거래를 학습에서 제외
- [x] Early stopping은 해당 Fold 검증 데이터에만 적용
- [x] Test는 전체 Train으로 fit한 전처리기의 transform만 적용

여러 Fold를 이용한 파라미터 선택 자체는 모델 선택 편향을 만들 수 있습니다. 최종 성능 추정이 특히 중요하다면 가장 최근 기간을 완전히 손대지 않은 최종 Holdout으로 별도 보관합니다.